# Whole-slide StarDist at level 0, processed in chunks

This proof of concept completely bypasses the low-resolution tissue-mask gate. Every level-0 chunk is read and sent to StarDist. A halo prevents boundary truncation during inference, and centre ownership prevents duplicate nuclei. No nucleus QC is applied.

The notebook checkpoints each chunk, so it can resume after interruption. Start with `MAX_CHUNKS = 6`; after checking the diagnostic images, set it to `None` for the complete slide.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from csbdeep.utils import normalize
from mxtifffile import MxTiffFile
from skimage.measure import regionprops_table
from stardist.models import StarDist2D

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Configuration

In [ ]:
QPTIFF_PATH = Path('/full/path/to/your/image.qptiff')
OUTPUT_DIR = Path('./level0_chunked_poc')
CHANNEL = 'DAPI'

CHUNK_SIZE = 4096
HALO = 128
PROB_THRESH = 0.35
NMS_THRESH = 0.40
N_TILES = (2, 2)
PREVIEW_LEVEL = 5

# Use 6 initially. Set to None only after inspecting the first diagnostics.
MAX_CHUNKS = 6
SAVE_FULL_LABEL_ARRAYS = False  # True can consume substantial disk space.
SAVE_DIAGNOSTIC_FOR_NONEMPTY_CHUNKS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'chunks').mkdir(exist_ok=True)
(OUTPUT_DIR / 'diagnostics').mkdir(exist_ok=True)
(OUTPUT_DIR / 'labels').mkdir(exist_ok=True)

## 2. Open the QPTIFF and construct every level-0 chunk
There is no tissue-mask test in this chunk generator.

In [ ]:
slide = MxTiffFile(str(QPTIFF_PATH))

def resolve_channel(slide, requested):
    if isinstance(requested, int):
        return requested
    requested = str(requested).strip().lower()
    for key in ('biomarker', 'fluorophore'):
        for info in slide.channel_info:
            value = info.get(key)
            if value and str(value).strip().lower() == requested:
                return int(info['index'])
    raise ValueError(f'Channel {requested!r} not found. Available: {slide.channel_info}')

def squeeze_channel(array):
    array = np.asarray(array)
    if array.ndim == 3 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 2:
        raise ValueError(f'Expected a 2-D channel; received {array.shape}')
    return array

layer = resolve_channel(slide, CHANNEL)
level0_shape = tuple(map(int, slide.series[0].levels[0].pages[0].shape[:2]))
preview_shape = tuple(map(int, slide.series[0].levels[PREVIEW_LEVEL].pages[0].shape[:2]))

def make_chunks(shape, chunk_size, halo):
    height, width = shape
    chunks = []
    chunk_id = 0
    for core_y0 in range(0, height, chunk_size):
        for core_x0 in range(0, width, chunk_size):
            core_y1 = min(height, core_y0 + chunk_size)
            core_x1 = min(width, core_x0 + chunk_size)
            chunks.append({
                'chunk_id': chunk_id,
                'core_y0': core_y0, 'core_x0': core_x0,
                'core_y1': core_y1, 'core_x1': core_x1,
                'read_y0': max(0, core_y0 - halo),
                'read_x0': max(0, core_x0 - halo),
                'read_y1': min(height, core_y1 + halo),
                'read_x1': min(width, core_x1 + halo),
            })
            chunk_id += 1
    return chunks

chunks = make_chunks(level0_shape, CHUNK_SIZE, HALO)
print('DAPI layer:', layer)
print('Level-0 shape:', level0_shape)
print('Total chunks:', len(chunks))
print('Chunks in this run:', len(chunks) if MAX_CHUNKS is None else min(MAX_CHUNKS, len(chunks)))

## 3. Load StarDist and define measurement
All detected shapes are retained. Ownership is not QC; it only prevents double counting across halos.

In [ ]:
model = StarDist2D.from_pretrained('2D_versatile_fluo')
PROPERTIES = [
    'label', 'area', 'perimeter', 'centroid', 'eccentricity', 'solidity',
    'major_axis_length', 'minor_axis_length', 'orientation',
    'mean_intensity', 'max_intensity'
]

def segment_measure_and_own(image, chunk):
    normalised = normalize(image.astype(np.float32), 1, 99.8, axis=(0, 1))
    labels, details = model.predict_instances(
        normalised, prob_thresh=PROB_THRESH, nms_thresh=NMS_THRESH, n_tiles=N_TILES
    )
    table = pd.DataFrame(regionprops_table(
        labels, intensity_image=normalised, properties=PROPERTIES
    ))
    if table.empty:
        return table, labels, normalised
    table = table.rename(columns={
        'centroid-0': 'local_y', 'centroid-1': 'local_x',
        'mean_intensity': 'mean_dapi', 'max_intensity': 'max_dapi'
    })
    table['y'] = table['local_y'] + chunk['read_y0']
    table['x'] = table['local_x'] + chunk['read_x0']
    owned = (
        (table['y'] >= chunk['core_y0']) & (table['y'] < chunk['core_y1']) &
        (table['x'] >= chunk['core_x0']) & (table['x'] < chunk['core_x1'])
    )
    table = table.loc[owned].copy()
    table['circularity'] = np.where(
        table['perimeter'] > 0,
        4 * np.pi * table['area'] / np.square(table['perimeter']), np.nan
    )
    table['aspect_ratio'] = np.where(
        table['minor_axis_length'] > 0,
        table['major_axis_length'] / table['minor_axis_length'], np.nan
    )
    table['chunk_id'] = chunk['chunk_id']
    return table, labels, normalised

## 4. Process or resume chunks
A chunk CSV is written atomically after completion. Re-running this cell skips completed chunks.

In [ ]:
run_chunks = chunks if MAX_CHUNKS is None else chunks[:MAX_CHUNKS]
for position, chunk in enumerate(run_chunks, start=1):
    csv_path = OUTPUT_DIR / 'chunks' / f"nuclei_{chunk['chunk_id']:06d}.csv"
    if csv_path.exists():
        print(f"[{position}/{len(run_chunks)}] reuse chunk {chunk['chunk_id']}")
        continue
    print(f"[{position}/{len(run_chunks)}] segment chunk {chunk['chunk_id']}")
    image = squeeze_channel(slide.read_region(
        layer,
        pos=(chunk['read_x0'], chunk['read_y0']),
        shape=(chunk['read_x1'] - chunk['read_x0'], chunk['read_y1'] - chunk['read_y0']),
        level=0,
    ))
    nuclei, labels, normalised = segment_measure_and_own(image, chunk)
    temporary = csv_path.with_suffix('.csv.tmp')
    nuclei.to_csv(temporary, index=False)
    temporary.replace(csv_path)

    if SAVE_FULL_LABEL_ARRAYS:
        np.savez_compressed(OUTPUT_DIR / 'labels' / f"labels_{chunk['chunk_id']:06d}.npz", labels=labels)

    if SAVE_DIAGNOSTIC_FOR_NONEMPTY_CHUNKS and not nuclei.empty:
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(normalised, cmap='gray')
        ax.scatter(nuclei['local_x'], nuclei['local_y'], s=5, facecolors='none',
                   edgecolors='#00ff66', linewidths=0.35)
        ax.set_title(f"Chunk {chunk['chunk_id']}: {len(nuclei):,} owned nuclei")
        ax.axis('off')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / 'diagnostics' / f"chunk_{chunk['chunk_id']:06d}.png", dpi=150)
        plt.close(fig)
    del image, labels, normalised
print('Finished requested chunks.')

## 5. Combine every completed nucleus table

In [ ]:
frames = []
for path in sorted((OUTPUT_DIR / 'chunks').glob('nuclei_*.csv')):
    try:
        frame = pd.read_csv(path)
        if not frame.empty:
            frames.append(frame)
    except pd.errors.EmptyDataError:
        pass
if not frames:
    raise RuntimeError('No nuclei found in completed chunks.')
nuclei = pd.concat(frames, ignore_index=True)
nuclei.to_csv(OUTPUT_DIR / 'level0_all_detected_nuclei.csv', index=False)
print('Completed chunks:', len(list((OUTPUT_DIR / 'chunks').glob('nuclei_*.csv'))))
print('Total owned nuclei:', len(nuclei))
display(nuclei[['area', 'circularity', 'eccentricity', 'solidity', 'aspect_ratio']].describe())

## 6. Whole-slide coverage preview
Each cyan point is one level-0 StarDist nucleus. This is not a grid and it does not use a tissue mask.

In [ ]:
preview = squeeze_channel(slide.read_region(layer, level=PREVIEW_LEVEL))
scale_x = preview.shape[1] / level0_shape[1]
scale_y = preview.shape[0] / level0_shape[0]

fig, ax = plt.subplots(figsize=(14, 18))
ax.imshow(preview, cmap='gray')
ax.scatter(nuclei['x'] * scale_x, nuclei['y'] * scale_y, s=1.0, c='#00e5ff', alpha=0.65)
ax.set_title(f'Level-0 StarDist coverage: {len(nuclei):,} nuclei')
ax.axis('off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'level0_whole_slide_nucleus_coverage.png', dpi=200, bbox_inches='tight')
plt.show()

## How to use this proof of concept

1. Run six chunks first and inspect `diagnostics/`.
2. If segmentation is reasonable, change `MAX_CHUNKS = None` and rerun the processing cell. Existing chunks resume automatically.
3. The first six chunks may be background because chunks are processed in row-major order. To test a known area first, replace `run_chunks = ...` with `run_chunks = [chunks[YOUR_CHUNK_ID]]`.
4. This notebook intentionally processes background chunks. Once coverage is verified, a nucleus-derived chunk map can replace the unreliable level-4/5 tissue gate.
5. The final cell-focused pipeline should cluster rows in `level0_all_detected_nuclei.csv`, then render each saved StarDist instance by phenotype.